# Results: Clustering and Visualization Metrics

This notebook consolidates all results for the final report. It loads precomputed CSVs from  and produces summary tables for:

- **Clustering metrics** (FM index, Rand, ARI, AMI, Dendrogram Purity, LCA-F1) across all DR + clustering method combinations for benchmark and synthetic datasets
- **Visualization quality metrics** (Trustworthiness, Continuity, Spearman Correlation, DEMaP) for all DR methods, split by embedding model

All inputs are precomputed - no pipeline steps need to be rerun. See [REPRODUCIBILITY.md](../../REPRODUCIBILITY.md) for instructions on regenerating the underlying data.

**Data sources:**
- Clustering CSVs:  and 
- Viz metric CSVs: 

In [1]:
import pandas as pd

datasets = {
    "arxiv":   "../../results/clustering/benchmark/arxiv_clustering_scores.csv",
    "rcv1":    "../../results/clustering/benchmark/rcv1_clustering_scores.csv",
    "wos":     "../../results/clustering/benchmark/wos_clustering_scores.csv",
    "dbpedia": "../../results/clustering/benchmark/db_clustering_scores.csv",
}

dfs = []
for name, path in datasets.items():
    df = pd.read_csv(path)
    df["dataset"] = name
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)
all_df = all_df.drop(columns=["TED"], errors="ignore")
all_df.head()

,embedding_model,reduction_method,cluster_method,level,FM,Rand,ARI,AMI,Dendrogram_Purity,LCA_F1,dataset
0,Qwen/Qwen3-Embedding-0.6B,PCA,Agglomerative,2,0.664852,0.604832,0.208450,0.239177,0.861195,0.687480,arxiv
1,Qwen/Qwen3-Embedding-0.6B,PCA,Agglomerative,61,0.200452,0.932777,0.146290,0.448880,0.398516,0.388156,arxiv
2,Qwen/Qwen3-Embedding-0.6B,PCA,DC,2,0.725925,0.594872,0.000835,-0.000035,0.861663,0.676537,arxiv
3,Qwen/Qwen3-Embedding-0.6B,PCA,DC,61,0.205915,0.932372,0.152697,0.457209,0.336575,0.328829,arxiv
4,Qwen/Qwen3-Embedding-0.6B,PCA,HDBSCAN,2,0.810123,0.656322,0.000169,0.000125,0.840655,0.696964,arxiv


In [2]:
results_main = {}

for name, file in datasets.items():
    df = pd.read_csv(file)
    df = df.drop(columns=["TED"], errors="ignore")
    df = df[df["level"] == 61]
    df = df.drop(columns=["level"], errors="ignore")

    df_summary = (
        df.groupby(["embedding_model", "reduction_method", "cluster_method"])
        .median(numeric_only=True)
        .reset_index()
    )

    df_summary["dataset"] = name
    results_main[name] = df_summary

In [3]:
# Stack all per-dataset summaries into a single DataFrame.
# Rows: one per (embedding model, DR method, cluster method, dataset) combo.
all_summary = pd.concat(results_main.values(), ignore_index=True)
all_summary


,embedding_model,reduction_method,cluster_method,FM,Rand,ARI,AMI,Dendrogram_Purity,LCA_F1,dataset
0,Qwen/Qwen3-Embedding-0.6B,PCA,Agglomerative,0.200452,0.932777,0.146290,0.448880,0.398516,0.388156,arxiv
1,Qwen/Qwen3-Embedding-0.6B,PCA,DC,0.205915,0.932372,0.152697,0.457209,0.336575,0.328829,arxiv
2,Qwen/Qwen3-Embedding-0.6B,PCA,HDBSCAN,0.248227,0.065433,0.000374,0.001552,0.191639,0.260146,arxiv
3,Qwen/Qwen3-Embedding-0.6B,PHATE,Agglomerative,0.190152,0.930400,0.139733,0.431202,0.413192,0.395964,arxiv
4,Qwen/Qwen3-Embedding-0.6B,PHATE,DC,0.194530,0.929954,0.144710,0.406288,0.308254,0.324004,arxiv
5,Qwen/Qwen3-Embedding-0.6B,PHATE,HDBSCAN,0.247192,0.073276,0.000415,0.005079,0.228875,0.285046,arxiv
6,Qwen/Qwen3-Embedding-0.6B,PaCMAP,Agglomerative,0.253098,0.936653,0.191981,0.500539,0.496534,0.442485,arxiv
7,Qwen/Qwen3-Embedding-0.6B,PaCMAP,DC,0.215554,0.933830,0.159615,0.471015,0.349514,0.328673,arxiv
8,Qwen/Qwen3-Embedding-0.6B,PaCMAP,HDBSCAN,0.291796,0.386082,0.055412,0.215360,0.297358,0.308074,arxiv
9,Qwen/Qwen3-Embedding-0.6B,Raw,Agglomerative,0.194963,0.932571,0.141105,0.437312,0.385721,0.368506,arxiv


## Visualization Metrics

Tables showing Trustworthiness, Continuity, Spearman Correlation, and DEMaP for each DR method, split by embedding model and dataset type.

In [4]:
import pandas as pd
import os
import glob

# ── Paths & constants

# Viz metrics live in results/viz_metrics/{embedding_model}/
VIZ_BASE = '../../results/viz_metrics'

BENCH_DATASETS = ['amazon', 'arxiv', 'dbpedia', 'rcv1', 'wos']

# Column names differ between benchmark (subsampled → mean/std stored) and
# synthetic (single-run → raw values) CSVs, so we track both sets.
METRICS_BENCH = ['Trustworthiness_mean', 'Continuity_mean',
                 'Spearman_Correlation_mean', 'DEMaP_mean']
METRIC_LABELS = ['Trustworthiness', 'Continuity', 'Spearman', 'DEMaP']
METRICS_SYNTH = ['Trustworthiness', 'Continuity', 'Spearman Correlation', 'DEMaP']


#  Loaders

def load_benchmark_viz(emb_dir):
    """Load all 5 benchmark viz-metric CSVs for one embedding model.
    Placeholder values (___) from incomplete runs are coerced to NaN
    so they are silently skipped in mean/std calculations.
    """
    frames = []
    for ds in BENCH_DATASETS:
        df = pd.read_csv(
            os.path.join(VIZ_BASE, emb_dir, f'viz_metrics_{ds}.csv'),
            na_values=['___'],   # treat ___ placeholders as missing
        )
        df['dataset'] = ds
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def load_synthetic_viz(emb_dir):
    """Load all synthetic-hierarchy viz-metric CSVs for one embedding model."""
    files = sorted(glob.glob(
        os.path.join(VIZ_BASE, emb_dir, 'viz_metrics_*hierarchy*.csv')
    ))
    frames = []
    for f in files:
        df = pd.read_csv(f)
        # Use the filename (minus prefix/suffix) as the dataset identifier.
        name = os.path.basename(f).replace('viz_metrics_', '').replace('.csv', '')
        df['dataset'] = name
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


#  Label helpers

def short_synth_label(name):
    """
    Turn a long synthetic dataset filename into a compact label, e.g.
      Energy_Ecosystems_and_Humans_hierarchy_t1.0_maxsub3_depth5_synonyms0_noise0.25_random
      → EEH_3s_d5_n25
    Topic:   EEH = Energy/Ecosystems/Humans,  OEF = Offshore Energy/Fisheries
    Config:  3s_d5 = maxsub3 depth5,          5s_d3 = maxsub5 depth3
    Noise:   _clean / _n25 / _n50
    """
    topic  = 'EEH' if 'Energy_Ecosystems' in name else 'OEF'
    config = '3s_d5' if 'maxsub3_depth5' in name else '5s_d3'
    noise  = '_n50' if 'noise0.5' in name else ('_n25' if 'noise0.25' in name else '_clean')
    return f'{topic}_{config}{noise}'


#  Wide-format pivot tables
def make_benchmark_table(df):
    """
    Pivot benchmark viz metrics into a wide table.
    Rows = DR method.  Columns = MultiIndex (dataset, metric).
    """
    sub = df[['dataset', 'Method'] + METRICS_BENCH].copy()
    sub = sub.rename(columns=dict(zip(METRICS_BENCH, METRIC_LABELS)))
    pivot = sub.set_index(['Method', 'dataset']).unstack('dataset')
    # Swap so the top level is dataset name, not metric name.
    pivot.columns = pivot.columns.swaplevel()
    pivot = pivot.sort_index(axis=1, level=0)
    return pivot.round(4)


def make_synthetic_table(df):
    """
    Pivot synthetic viz metrics into a wide table.
    Rows = DR method.  Columns = MultiIndex (short config label, metric).
    """
    df2 = df.copy()
    df2['config'] = df2['dataset'].apply(short_synth_label)
    sub = df2[['Method', 'config'] + METRICS_SYNTH].rename(
        columns={'Spearman Correlation': 'Spearman'}
    )
    pivot = sub.set_index(['Method', 'config']).unstack('config')
    pivot.columns = pivot.columns.swaplevel()
    pivot = pivot.sort_index(axis=1, level=0)
    return pivot.round(4)


def make_summary_table(bench_df, synth_df):
    """
    Mean across all datasets for both benchmark and synthetic,
    side-by-side in one table.  Used for a quick overall comparison.
    """
    # Benchmark: average the per-dataset means across the 5 benchmark datasets.
    bench_mean = bench_df.groupby('Method')[METRICS_BENCH].mean().round(4)
    bench_mean.columns = METRIC_LABELS
    bench_mean.columns = pd.MultiIndex.from_product([['Benchmark'], bench_mean.columns])

    # Synthetic: average across all 12 configs (2 topics × 2 structures × 3 noise levels).
    synth_sub  = synth_df.rename(columns={'Spearman Correlation': 'Spearman'})
    synth_mean = synth_sub.groupby('Method')[
        ['Trustworthiness', 'Continuity', 'Spearman', 'DEMaP']
    ].mean().round(4)
    synth_mean.columns = pd.MultiIndex.from_product([['Synthetic'], synth_mean.columns])

    return pd.concat([bench_mean, synth_mean], axis=1)


#  Load data

qwen_bench = load_benchmark_viz('Qwen')
st_bench   = load_benchmark_viz('sentence-transformers')
qwen_synth = load_synthetic_viz('Qwen')
st_synth   = load_synthetic_viz('sentence-transformers')

print("Benchmark files per model:", len(qwen_bench['dataset'].unique()), "datasets")
print("Synthetic configs per model:", len(qwen_synth['dataset'].unique()), "configs")


Benchmark files per model: 5 datasets
Synthetic configs per model: 12 configs


### Benchmark Datasets - Qwen (`Qwen3-Embedding-0.6B`)

In [5]:
# Wide pivot table: rows = DR method, columns = (dataset, metric).
# Each cell is the mean viz-metric value for that DR × dataset pair.
make_benchmark_table(qwen_bench)


dataset     amazon                                       arxiv          \
        Continuity   DEMaP Spearman Trustworthiness Continuity   DEMaP   
Method                                                                   
PCA         0.9006  0.5186   0.4495          0.7682     0.8776  0.4659   
PHATE       0.9342  0.6284   0.4498          0.8368     0.9177  0.6205   
PaCMAP      0.9306  0.5491   0.4178          0.9044     0.9178  0.5785   
TriMAP      0.9399  0.5823   0.4405          0.8979     0.9263  0.5611   
UMAP        0.9336  0.5564   0.4115          0.9325     0.9210  0.5997   
tSNE        0.9343  0.5431   0.4516          0.9550     0.8999  0.4029   

dataset                             dbpedia                                   \
        Spearman Trustworthiness Continuity   DEMaP Spearman Trustworthiness   
Method                                                                         
PCA       0.5039          0.7503     0.8802  0.3434   0.3584          0.7209   
PHATE     0.5377          0.8050     0.9454  0.5720   0.3250          0.8953   
PaCMAP    0.5015          0.9099     0.9453  0.4262   0.3073          0.9631   
TriMAP    0.5078          0.8860     0.9507  0.5133   0.3013          0.9568   
UMAP      0.4948          0.9354     0.9346  0.5033   0.2658          0.9740   
tSNE      0.3656          0.9599     0.9087  0.3776   0.2500          0.9660   

dataset       rcv1                                         wos          \
        Continuity   DEMaP Spearman Trustworthiness Continuity   DEMaP   
Method                                                                   
PCA         0.8937  0.4907   0.5160          0.7921     0.9092  0.5668   
PHATE       0.8925  0.5793   0.3098          0.8074     0.9424  0.6909   
PaCMAP      0.9350  0.6029   0.3959          0.9466     0.9434  0.6282   
TriMAP      0.9313  0.6246   0.3602          0.9404     0.9463  0.6661   
UMAP        0.9373  0.6743   0.4002          0.9621     0.9388  0.6409   
tSNE        0.9364  0.5187   0.4244          0.9621     0.9391  0.6006   

dataset                           
        Spearman Trustworthiness  
Method                            
PCA       0.4794          0.7749  
PHATE     0.4452          0.8488  
PaCMAP    0.4322          0.9240  
TriMAP    0.4343          0.8957  
UMAP      0.4121          0.9495  
tSNE      0.4181          0.9752

### Benchmark Datasets - sentence-transformers (`all-MiniLM-L6-v2`)

In [6]:
# Same layout as above, for the sentence-transformers (MiniLM) embedding model.
make_benchmark_table(st_bench)


dataset     amazon                                       arxiv          \
        Continuity   DEMaP Spearman Trustworthiness Continuity   DEMaP   
Method                                                                   
PCA         0.8636  0.3323   0.3639          0.7392     0.8936  0.4621   
PHATE       0.9082  0.4932   0.3441          0.8022     0.9354  0.7014   
PaCMAP      0.9071  0.4536   0.3571          0.8645     0.9342  0.6530   
TriMAP      0.9128  0.4416   0.3448          0.8680     0.9429  0.6504   
UMAP        0.9211  0.5220   0.4077          0.9114     0.9301  0.6275   
tSNE        0.8549  0.3345   0.2923          0.9096     0.9310  0.5205   

dataset                             dbpedia                                   \
        Spearman Trustworthiness Continuity   DEMaP Spearman Trustworthiness   
Method                                                                         
PCA       0.5018          0.7607     0.8707  0.2524   0.3033          0.7118   
PHATE     0.6377          0.8165     0.9382  0.5202   0.2641          0.8816   
PaCMAP    0.5770          0.9103     0.9370  0.4064   0.2360          0.9600   
TriMAP    0.5848          0.8871     0.9456  0.4915   0.2780          0.9469   
UMAP      0.5318          0.9440     0.9424  0.5045   0.2744          0.9749   
tSNE      0.4606          0.9704     0.5585 -0.0114   0.0094          0.5752   

dataset       rcv1                                         wos          \
        Continuity   DEMaP Spearman Trustworthiness Continuity   DEMaP   
Method                                                                   
PCA         0.8913  0.5048   0.3965          0.7781     0.8940  0.5663   
PHATE       0.9063  0.5745   0.3821          0.8233     0.9395  0.7213   
PaCMAP      0.9333  0.6321   0.4219          0.9360     0.9441  0.6660   
TriMAP      0.9325  0.5848   0.3723          0.9357     0.9449  0.6920   
UMAP        0.9426  0.6601   0.4586          0.9598     0.9319  0.6208   
tSNE        0.9432  0.6035   0.4597          0.9560     0.9263  0.5626   

dataset                           
        Spearman Trustworthiness  
Method                            
PCA       0.5429          0.7861  
PHATE     0.5612          0.8614  
PaCMAP    0.5111          0.9276  
TriMAP    0.5350          0.9071  
UMAP      0.4660          0.9466  
tSNE      0.4292          0.9751

### Synthetic Datasets - Qwen (`Qwen3-Embedding-0.6B`)

Config labels - topic: `EEH` = Energy/Ecosystems/Humans, `OEF` = Offshore Energy/Fisheries; structure: `3s_d5` = maxsub3 depth5, `5s_d3` = maxsub5 depth3; noise: `clean` / `n25` (25%) / `n50` (50%).

In [7]:
# Wide pivot table: rows = DR method, columns = (short config label, metric).
# Config labels encode topic, tree structure, and noise level (see short_synth_label).
make_synthetic_table(qwen_synth)


config EEH_3s_d5_clean                                  EEH_3s_d5_n25          \
            Continuity   DEMaP Spearman Trustworthiness    Continuity   DEMaP   
Method                                                                          
PCA             0.9082  0.5089   0.5710          0.8018        0.9097  0.2818   
PHATE           0.9250  0.6326   0.4756          0.8461        0.9317  0.6162   
PaCMAP          0.9548  0.6212   0.5185          0.9636        0.9538  0.4115   
TriMAP          0.9594  0.6154   0.5150          0.9617        0.9592  0.4488   
UMAP            0.9648  0.7081   0.5701          0.9756        0.9634  0.4946   
tSNE            0.9615  0.5990   0.4704          0.9745        0.9614  0.5391   

config                          EEH_3s_d5_n50          ... OEF_5s_d3_clean  \
       Spearman Trustworthiness    Continuity   DEMaP  ...        Spearman   
Method                                                 ...                   
PCA      0.4278          0.7953        0.9128  0.2734  ...          0.5329   
PHATE    0.4689          0.8440        0.9494  0.5616  ...          0.4689   
PaCMAP   0.3842          0.9602        0.9434  0.4029  ...          0.3175   
TriMAP   0.4229          0.9552        0.9483  0.3854  ...          0.2604   
UMAP     0.4153          0.9718        0.9565  0.4917  ...          0.3961   
tSNE     0.4530          0.9715        0.9525  0.4265  ...          0.4962   

config                 OEF_5s_d3_n25                                   \
       Trustworthiness    Continuity   DEMaP Spearman Trustworthiness   
Method                                                                  
PCA             0.7553        0.8822  0.1961   0.4125          0.7502   
PHATE           0.8261        0.9186  0.5170   0.3905          0.8062   
PaCMAP          0.9401        0.9131  0.2922   0.2366          0.9329   
TriMAP          0.9362        0.9203  0.2444   0.1946          0.9377   
UMAP            0.9414        0.9353  0.4296   0.3152          0.9414   
tSNE            0.7217        0.9278  0.2623   0.3004          0.8413   

config OEF_5s_d3_n50                                   
          Continuity   DEMaP Spearman Trustworthiness  
Method                                                 
PCA           0.8935  0.4264   0.5397          0.7779  
PHATE         0.8977  0.5419   0.3857          0.7764  
PaCMAP        0.9302  0.4688   0.4004          0.9396  
TriMAP        0.9346  0.4938   0.3918          0.9457  
UMAP          0.9422  0.5909   0.3485          0.9454  
tSNE          0.9409  0.5477   0.5711          0.9480  

[6 rows x 48 columns]

### Synthetic Datasets - sentence-transformers (`all-MiniLM-L6-v2`)

In [8]:
# Same layout as above, for the sentence-transformers (MiniLM) embedding model.
make_synthetic_table(st_synth)


config EEH_3s_d5_clean                                  EEH_3s_d5_n25          \
            Continuity   DEMaP Spearman Trustworthiness    Continuity   DEMaP   
Method                                                                          
PCA             0.8953  0.4164   0.5318          0.7917        0.8972  0.1903   
PHATE           0.9115  0.5282   0.4434          0.8292        0.9442  0.5704   
PaCMAP          0.9435  0.5384   0.4876          0.9567        0.9535  0.3801   
TriMAP          0.9529  0.5186   0.4730          0.9519        0.9526  0.3263   
UMAP            0.9582  0.5999   0.5100          0.9700        0.9577  0.3230   
tSNE            0.9592  0.5753   0.4811          0.9644        0.9601  0.4132   

config                          EEH_3s_d5_n50          ... OEF_5s_d3_clean  \
       Spearman Trustworthiness    Continuity   DEMaP  ...        Spearman   
Method                                                 ...                   
PCA      0.3488          0.7816        0.9039  0.1927  ...          0.4614   
PHATE    0.4422          0.8556        0.9503  0.4449  ...          0.4023   
PaCMAP   0.3828          0.9531        0.9462  0.3388  ...          0.3638   
TriMAP   0.3367          0.9518        0.9484  0.2709  ...          0.3647   
UMAP     0.2597          0.9609        0.9561  0.3686  ...          0.3387   
tSNE     0.4048          0.9672        0.9538  0.3721  ...          0.4469   

config                 OEF_5s_d3_n25                                   \
       Trustworthiness    Continuity   DEMaP Spearman Trustworthiness   
Method                                                                  
PCA             0.7611        0.8712  0.3342   0.4575          0.7357   
PHATE           0.8135        0.9130  0.5860   0.2375          0.7804   
PaCMAP          0.9389        0.9185  0.3713   0.1385          0.9284   
TriMAP          0.9307        0.9231  0.3311   0.1527          0.9319   
UMAP            0.9424        0.9325  0.4711   0.2136          0.9367   
tSNE            0.7171        0.9228  0.4095   0.3880          0.9371   

config OEF_5s_d3_n50                                   
          Continuity   DEMaP Spearman Trustworthiness  
Method                                                 
PCA           0.8827  0.4458   0.4575          0.7309  
PHATE         0.9068  0.5483   0.1924          0.7447  
PaCMAP        0.9252  0.4208   0.2597          0.9202  
TriMAP        0.9317  0.3940   0.2027          0.9301  
UMAP          0.9417  0.4713   0.2278          0.9314  
tSNE          0.9347  0.4738   0.4013          0.9329  

[6 rows x 48 columns]

## Summary Tables: Mean ± Std Across Datasets

Each cell shows `mean ± std` computed across datasets for a given DR method and metric. Benchmark: mean ± std over 5 datasets (amazon, arxiv, dbpedia, rcv1, wos). Synthetic: mean ± std over 12 configs (2 topics × 3 noise levels × 2 tree structures).

In [9]:
def make_mean_std_table(df, is_benchmark):
    """
    Summarise viz metrics as 'mean ± std' across datasets.

    For benchmark  (is_benchmark=True):  std over 5 benchmark datasets.
    For synthetic  (is_benchmark=False): std over 12 synthetic configs.

    Columns returned: T (Trustworthiness), C (Continuity), Spearman, DEMaP.
    """
    # Pick the right source column names depending on data type.
    if is_benchmark:
        src_cols = ['Trustworthiness_mean', 'Continuity_mean',
                    'Spearman_Correlation_mean', 'DEMaP_mean']
    else:
        src_cols = ['Trustworthiness', 'Continuity', 'Spearman Correlation', 'DEMaP']

    dst_cols = ['T', 'C', 'Spearman', 'DEMaP']

    # Rename to short display names before grouping.
    sub = df[['Method'] + src_cols].rename(columns=dict(zip(src_cols, dst_cols)))

    grp  = sub.groupby('Method')[dst_cols]
    mean = grp.mean()
    std  = grp.std()

    # Format each cell as "mean ± std" string.
    out = pd.DataFrame(index=mean.index)
    for col in dst_cols:
        out[col] = mean[col].map('{:.4f}'.format) + ' \u00b1 ' + std[col].map('{:.4f}'.format)

    return out


### Benchmark Datasets - all-MiniLM-L6-v2

In [10]:
# Mean ± std across 5 benchmark datasets - sentence-transformers (MiniLM).
make_mean_std_table(st_bench, is_benchmark=True)


,T,C,Spearman,DEMaP
Method,,,,
PCA,0.7552 ± 0.0302,0.8826 ± 0.0144,0.4217 ± 0.0989,0.4236 ± 0.1285
PHATE,0.8370 ± 0.0332,0.9255 ± 0.0168,0.4378 ± 0.1559,0.6021 ± 0.1042
PaCMAP,0.9197 ± 0.0357,0.9311 ± 0.0141,0.4206 ± 0.1330,0.5622 ± 0.1224
TriMAP,0.9090 ± 0.0328,0.9357 ± 0.0139,0.4230 ± 0.1308,0.5721 ± 0.1051
UMAP,0.9473 ± 0.0235,0.9336 ± 0.0091,0.4277 ± 0.0964,0.5870 ± 0.0692
tSNE,0.8773 ± 0.1708,0.8428 ± 0.1626,0.3302 ± 0.1923,0.4019 ± 0.2529


### Benchmark Datasets - Qwen3-Embedding-0.6B

In [11]:
# Mean ± std across 5 benchmark datasets - Qwen3-Embedding-0.6B.
make_mean_std_table(qwen_bench, is_benchmark=True)


,T,C,Spearman,DEMaP
Method,,,,
PCA,0.7613 ± 0.0271,0.8923 ± 0.0134,0.4614 ± 0.0630,0.4771 ± 0.0836
PHATE,0.8387 ± 0.0368,0.9264 ± 0.0218,0.4135 ± 0.0953,0.6182 ± 0.0475
PaCMAP,0.9296 ± 0.0248,0.9344 ± 0.0111,0.4109 ± 0.0701,0.5570 ± 0.0788
TriMAP,0.9154 ± 0.0312,0.9389 ± 0.0101,0.4088 ± 0.0796,0.5895 ± 0.0587
UMAP,0.9507 ± 0.0176,0.9331 ± 0.0071,0.3969 ± 0.0825,0.5949 ± 0.0677
tSNE,0.9636 ± 0.0076,0.9237 ± 0.0180,0.3819 ± 0.0801,0.4886 ± 0.0950


### Synthetic Datasets - all-MiniLM-L6-v2

In [12]:
# Mean ± std across 12 synthetic configs - sentence-transformers (MiniLM).
make_mean_std_table(st_synth, is_benchmark=False)


,T,C,Spearman,DEMaP
Method,,,,
PCA,0.7623 ± 0.0211,0.8862 ± 0.0127,0.4334 ± 0.0724,0.3375 ± 0.1047
PHATE,0.8287 ± 0.0451,0.9261 ± 0.0238,0.3845 ± 0.0980,0.5927 ± 0.0773
PaCMAP,0.9399 ± 0.0108,0.9356 ± 0.0172,0.3435 ± 0.0973,0.4699 ± 0.1015
TriMAP,0.9394 ± 0.0083,0.9381 ± 0.0146,0.3059 ± 0.0881,0.4333 ± 0.0965
UMAP,0.9521 ± 0.0128,0.9466 ± 0.0145,0.3680 ± 0.0933,0.5350 ± 0.1119
tSNE,0.9150 ± 0.0919,0.9342 ± 0.0304,0.4172 ± 0.0378,0.4888 ± 0.0648


### Synthetic Datasets - Qwen3-Embedding-0.6B

In [13]:
# Mean ± std across 12 synthetic configs - Qwen3-Embedding-0.6B.
make_mean_std_table(qwen_synth, is_benchmark=False)


,T,C,Spearman,DEMaP
Method,,,,
PCA,0.7754 ± 0.0183,0.8948 ± 0.0143,0.4865 ± 0.0636,0.3780 ± 0.1136
PHATE,0.8459 ± 0.0348,0.9280 ± 0.0211,0.4600 ± 0.0425,0.6282 ± 0.0746
PaCMAP,0.9487 ± 0.0098,0.9391 ± 0.0156,0.4154 ± 0.0812,0.5091 ± 0.1075
TriMAP,0.9486 ± 0.0074,0.9432 ± 0.0160,0.3917 ± 0.0939,0.5187 ± 0.1341
UMAP,0.9580 ± 0.0114,0.9509 ± 0.0128,0.4442 ± 0.0769,0.6162 ± 0.1038
tSNE,0.9186 ± 0.0770,0.9418 ± 0.0221,0.4718 ± 0.0763,0.5121 ± 0.1013
